## 0.Librerias y Funciones

#### Librerias

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
pd.options.mode.chained_assignment = None 
import os
import sys
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import SGDRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestRegressor

pd.options.display.max_rows=100
pd.options.display.max_columns=500

In [2]:
os.getcwd()

'C:\\Users\\KEVIN\\Downloads\\DSRP\\Machine Learning III\\DSRP_MLE3\\notebooks'

## 1. Data Raw: Exploracion

In [3]:
RAW_DATA_DIR = "../data/raw/"
PROCESSED_DATA_DIR = "../data/processed/"

In [4]:
data_raw = pd.read_csv(RAW_DATA_DIR+"PS_20174392719_1491204439457_log.csv")
print(data_raw.shape)

(6362620, 11)


In [5]:
data_raw.head(100)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.00,160296.36,M1979787155,0.00,0.00,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.00,19384.72,M2044282225,0.00,0.00,0,0
2,1,TRANSFER,181.00,C1305486145,181.00,0.00,C553264065,0.00,0.00,1,0
3,1,CASH_OUT,181.00,C840083671,181.00,0.00,C38997010,21182.00,0.00,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.00,29885.86,M1230701703,0.00,0.00,0,0
5,1,PAYMENT,7817.71,C90045638,53860.00,46042.29,M573487274,0.00,0.00,0,0
6,1,PAYMENT,7107.77,C154988899,183195.00,176087.23,M408069119,0.00,0.00,0,0
7,1,PAYMENT,7861.64,C1912850431,176087.23,168225.59,M633326333,0.00,0.00,0,0
8,1,PAYMENT,4024.36,C1265012928,2671.00,0.00,M1176932104,0.00,0.00,0,0
9,1,DEBIT,5337.77,C712410124,41720.00,36382.23,C195600860,41898.00,40348.79,0,0


#### Valores

In [6]:
print("isFraud (Target):", data_raw['isFraud'].unique())
print("isFlaggedFraud:", data_raw['isFlaggedFraud'].unique())
print("type:", data_raw['type'].unique())

isFraud (Target): [0 1]
isFlaggedFraud: [0 1]
type: ['PAYMENT' 'TRANSFER' 'CASH_OUT' 'DEBIT' 'CASH_IN']


#### Entidades transaccion

In [7]:
data_raw['entTrxType'] = data_raw['nameOrig'].str[0] + data_raw['nameDest'].str[0]
data_raw['entTrxType'].unique()

array(['CM', 'CC'], dtype=object)

In [8]:
data_raw.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,CM
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,CM
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,CC
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,CC
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,CM


In [9]:
data_raw[data_raw['type'] == 'CASH_IN'].head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType
389,1,CASH_IN,143236.26,C1862994526,0.00,143236.26,C1688019098,608932.17,97263.78,0,0,CC
390,1,CASH_IN,228451.89,C1614133563,143236.26,371688.15,C2083562754,719678.38,1186556.81,0,0,CC
391,1,CASH_IN,35902.49,C839771540,371688.15,407590.65,C2001112025,49003.30,0.00,0,0,CC
392,1,CASH_IN,232953.64,C1037163664,407590.65,640544.28,C33524623,1172672.27,1517262.16,0,0,CC
393,1,CASH_IN,65912.95,C180316302,640544.28,706457.23,C1330106945,104198.26,24044.18,0,0,CC


#### Monto transaccion

In [10]:
data_raw['origTrxAmount'] = data_raw['newbalanceOrig'] - data_raw['oldbalanceOrg']
data_raw['destTrxAmount'] = np.where(data_raw['entTrxType'] == "CM", np.nan, data_raw['newbalanceDest'] - data_raw['oldbalanceDest'])
#se considera NaN para los casos "CM" para diferenciar los valores "CC" con valor 0.

In [11]:
data_raw.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType,origTrxAmount,destTrxAmount
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,CM,-9839.64,NaN
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,CM,-1864.28,NaN
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,CC,-181.00,0.0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,CC,-181.00,-21182.0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,CM,-11668.14,NaN


In [12]:
data_raw[data_raw['type'] == 'CASH_IN'].head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType,origTrxAmount,destTrxAmount
389,1,CASH_IN,143236.26,C1862994526,0.00,143236.26,C1688019098,608932.17,97263.78,0,0,CC,143236.26,-511668.39
390,1,CASH_IN,228451.89,C1614133563,143236.26,371688.15,C2083562754,719678.38,1186556.81,0,0,CC,228451.89,466878.43
391,1,CASH_IN,35902.49,C839771540,371688.15,407590.65,C2001112025,49003.30,0.00,0,0,CC,35902.50,-49003.30
392,1,CASH_IN,232953.64,C1037163664,407590.65,640544.28,C33524623,1172672.27,1517262.16,0,0,CC,232953.63,344589.89
393,1,CASH_IN,65912.95,C180316302,640544.28,706457.23,C1330106945,104198.26,24044.18,0,0,CC,65912.95,-80154.08


#### Flags de fraude

In [13]:
data_raw[data_raw['isFlaggedFraud'] == 1].head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType,origTrxAmount,destTrxAmount
2736446,212,TRANSFER,4953893.08,C728984460,4953893.08,4953893.08,C639921569,0.0,0.0,1,1,CC,0.0,0.0
3247297,250,TRANSFER,1343002.08,C1100582606,1343002.08,1343002.08,C1147517658,0.0,0.0,1,1,CC,0.0,0.0
3760288,279,TRANSFER,536624.41,C1035541766,536624.41,536624.41,C1100697970,0.0,0.0,1,1,CC,0.0,0.0
5563713,387,TRANSFER,4892193.09,C908544136,4892193.09,4892193.09,C891140444,0.0,0.0,1,1,CC,0.0,0.0
5996407,425,TRANSFER,10000000.00,C689608084,19585040.37,19585040.37,C1392803603,0.0,0.0,1,1,CC,0.0,0.0


In [14]:
data_raw[(data_raw['isFlaggedFraud'] == 1)&(data_raw['isFraud'] == 0)] #todos los registros isFlaggedFraud=1 estan marcados como isFraud=1

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType,origTrxAmount,destTrxAmount


#### Base

In [15]:
data_raw['base'] = np.where(data_raw['step'] <= 400, 0, 2) # 0=Desarrollo, 2=Produccion

In [16]:
data_raw.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud,entTrxType,origTrxAmount,destTrxAmount,base
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0,CM,-9839.64,NaN,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0,CM,-1864.28,NaN,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0,CC,-181.00,0.0,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0,CC,-181.00,-21182.0,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0,CM,-11668.14,NaN,0


In [6]:
df_train = data_raw[data_raw['base']==0]
df_train = df_train.drop(columns={'base'})
df_train.to_csv(RAW_DATA_DIR+"train.csv", index=False)
del df_train

In [ ]:
df_prod = data_raw[data_raw['base']==2]
df_prod = df_prod.drop(columns={'base'})
df_prod.to_csv(RAW_DATA_DIR+"prod.csv", index=False)
del df_prod

#### Target

In [32]:
data_raw['step'].value_counts().sort_index(ascending=True)

step
1      2708
2      1014
3       552
4       565
5       665
       ... 
739      10
740       6
741      22
742      14
743       8
Name: count, Length: 743, dtype: int64

In [35]:
data_raw.groupby('base').agg({'isFraud':'mean'})

,isFraud
base,
0,0.000774
2,0.006491


## 2. Data Processed: Export

In [17]:
data_raw = data_raw[data_raw['isFlaggedFraud'] == 0] # Solo se consideran los 0, puesto que esta marca es una decision de imputacion

In [18]:
cols_export = ['base','type','amount','entTrxType','oldbalanceOrg','origTrxAmount','oldbalanceDest','destTrxAmount','isFraud']
data_raw[cols_export].to_parquet(PROCESSED_DATA_DIR+"processed.parquet", engine='pyarrow', compression='GZIP')